<a href="https://colab.research.google.com/github/aakilkhanemon/AI-driven-marine-lead-discovery/blob/main/GNINA_X_3L_X_Aur.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install necessary python structural suites quietly
!pip install -q rdkit mdanalysis

# 2. Fetch and grant execution permissions to the GNINA binary
!wget -q https://github.com/gnina/gnina/releases/download/v1.1/gnina -O gnina
!chmod +x gnina

print("=== ENVIRONMENT SETUP COMPLETE ===")
print("GNINA binary and Python libraries are ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 155.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.2 MB/s eta 0:00:00
=== ENVIRONMENT SETUP COMPLETE ===
GNINA binary and Python libraries are ready.


In [3]:
!./gnina -r receptor.pdb -l ligand.sdf --center_x -24.980 --center_y -2.037 --center_z -27.028 --size_x 29 --size_y 29 --size_z 29 --num_modes 5 --device 0 --cnn_scoring rescore --out test_docked.sdf

Streaming output truncated to the last 5000 lines.
  but OpenBabel found '  ' (atom 458)
*** Open Babel Warning  in parseAtomRecord
  Problems reading a HETATM or ATOM record.
  According to the PDB specification,
  columns 77-78 should contain the element symbol of an atom.
  but OpenBabel found '  ' (atom 459)
*** Open Babel Warning  in parseAtomRecord
  Problems reading a HETATM or ATOM record.
  According to the PDB specification,
  columns 77-78 should contain the element symbol of an atom.
  but OpenBabel found '  ' (atom 460)
*** Open Babel Warning  in parseAtomRecord
  Problems reading a HETATM or ATOM record.
  According to the PDB specification,
  columns 77-78 should contain the element symbol of an atom.
  but OpenBabel found '  ' (atom 461)
*** Open Babel Warning  in parseAtomRecord
  Problems reading a HETATM or ATOM record.
  According to the PDB specification,
  columns 77-78 should contain the element symbol of an atom.
  but OpenBabel found '  ' (atom 462)
*** Open Ba

In [4]:
import os
from rdkit import Chem

# Target files from your Colab workspace sidebar
sdf_input = 'test_docked.sdf'
receptor_input = 'receptor.pdb'
complex_output = 'best_docked_complex.pdb'

if not os.path.exists(sdf_input) or not os.path.exists(receptor_input):
    print("❌ Error: Missing files in the sidebar. Run your GNINA command first.")
else:
    # Read the docked ligand file (automatically prioritizes Pose 1)
    supplier = Chem.SDMolSupplier(sdf_input)
    top_pose = next(supplier)

    if top_pose is not None:
        # Convert the structural coordinates of the ligand to PDB format block
        ligand_block = Chem.MolToPDBBlock(top_pose)

        # Parse the prepped protein receptor structure
        with open(receptor_input, 'r') as f:
            receptor_lines = f.readlines()

        # Clear legacy structural endpoints to avoid text truncation bugs
        clean_receptor = [line for line in receptor_lines if not line.startswith(('END', 'CONECT'))]

        # Merge frameworks cleanly into a standard coordinate file
        with open(complex_output, 'w') as f:
            f.writelines(clean_receptor)
            f.write("\n")  # Section buffer spacer
            f.write(ligand_block)

        print(f"✅ SUCCESS: Unified structural model saved as -> {complex_output}")
        print(f"⭐ Extracted Mode 1 Metadata | CNN Pose Score: {top_pose.GetProp('CNNscore')} | Vina Energy: {top_pose.GetProp('minimizedAffinity')} kcal/mol")

✅ SUCCESS: Unified structural model saved as -> best_docked_complex.pdb
⭐ Extracted Mode 1 Metadata | CNN Pose Score: 0.7131156921 | Vina Energy: -5.06274 kcal/mol


[21:57:33] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


In [6]:
# 1. Install py3Dmol quietly in your current Colab session
!pip install -q py3Dmol

import py3Dmol

# 2. Initialize the 3D canvas viewport
viewer = py3Dmol.view(width=800, height=600)

with open('best_docked_complex.pdb', 'r') as f:
    complex_data = f.read()

viewer.addModel(complex_data, 'pdb')

# Style the protein cartoon ribbon backbone
viewer.setStyle({'cartoon': {'color': 'spectrum'}})

# Render the extracted top ligand pose as distinct colored sticks
viewer.setStyle({'hetatm': {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.2}}})

# Highlight your 4 target pocket interface residues (Tyr89, Glu96, Glu97, Pro164)
viewer.addStyle({'resid': [89, 96, 97, 164], 'chain': 'A'}, {'stick': {'colorscheme': 'magentaCarbon'}})

viewer.zoomTo({'hetatm': {}})  # Dynamically anchor the camera frame on the ligand coordinates
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
# 1. Re-verify the package is present
!pip install -q py3Dmol

import py3Dmol
import os

complex_file = 'best_docked_complex.pdb'

if not os.path.exists(complex_file):
    print(f"❌ Error: {complex_file} not found. Please re-run the file assembly step first!")
else:
    # 2. Re-initialize the canvas entirely within this single cell
    viewer = py3Dmol.view(width=800, height=600)

    with open(complex_file, 'r') as f:
        complex_data = f.read()

    # 3. Load the data stream into the viewport frame
    viewer.addModel(complex_data, 'pdb')

    # 4. Apply all consecutive architectural styling flags
    viewer.setStyle({'cartoon': {'color': 'spectrum'}})  # Protein ribbon
    viewer.setStyle({'hetatm': {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.2}}})  # Ligand Mode 1

    # Highlight your target interface residues (Tyr89, Glu96, Glu97, Pro164)
    viewer.addStyle({'resid': [89, 96, 97, 164], 'chain': 'A'}, {'stick': {'colorscheme': 'magentaCarbon'}})

    # 5. Snap the camera focus and render the active WebGL window
    viewer.zoomTo({'hetatm': {}})
    viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.